In [1]:
import numpy as np
import pandas as pd
from collections import defaultdict

In [2]:
df = pd.read_csv('blackjack_simulator.csv', nrows=100000)
# df = pd.read_csv("blackjack_simulator.csv") #Entire dataset
df.head()

,shoe_id,cards_remaining,dealer_up,initial_hand,dealer_final,dealer_final_value,player_final,player_final_value,actions_taken,run_count,true_count,win
0,0,416,10,"[10, 11]","[10, 4, 10]",24,"[[10, 11]]",['BJ'],[['S']],1,0,1.5
1,0,411,10,"[5, 5]","[10, 8]",18,"[[5, 5, 11]]",[21],"[['H', 'S']]",-2,0,1.0
2,0,406,6,"[3, 10]","[6, 6, 10]",22,"[[3, 10]]",[13],[['S']],-2,0,1.0
3,0,401,10,"[5, 9]","[10, 8]",18,"[[5, 9, 11, 3]]",[18],"[['H', 'H', 'S']]",-1,0,0.0
4,0,395,8,"[6, 10]","[8, 2, 10]",20,"[[6, 10, 10]]",[26],[['H']],-1,0,-1.0


## State Definition

In [3]:
def get_player_hand_value(hand):
    """Calculates the best possible hand value (accounting for Aces)."""
    hand = hand[0]
    ace_count = hand.count(11)
    total = sum(hand)
    while total > 21 and ace_count > 0:
        total -= 10
        ace_count -= 1
    return total


def has_usable_ace(hand):
    """Checks if the hand contains a usable Ace (can be 11 without busting)."""
    hand = hand[0]
    return 11 in hand and sum(hand) <= 21


def get_state(row):
    """Defines the state based on the current row of the dataframe."""
    player_hand = eval(row['player_final'])
    player_hand_value = get_player_hand_value(player_hand)
    usable_ace = has_usable_ace(player_hand)
    dealer_upcard = row['dealer_up']
    true_count = row['true_count']

    return (player_hand_value, dealer_upcard, usable_ace, true_count)

In [4]:
df['state'] = df.apply(get_state, axis=1)

## Action Space

In [5]:
ACTION_SPACE = ['H', 'S']
ACTION_MAP = {'H': 0, 'S': 1}
REVERSE_ACTION_MAP = {0: 'H', 1: 'S'}

## Reward Function

In [6]:
def get_reward(row):
    """Assigns a reward based on the outcome of the hand."""
    win = row['win']
    if win > 0:  # Won the hand
        return win
    elif win < 0:  # Lost the hand
        return win
    else:  # Push (tie)
        return 0

In [7]:
df['reward'] = df.apply(get_reward, axis=1)

## Policy

In [8]:
def create_q_table():
    """Initializes the Q-table."""
    q_table = defaultdict(lambda: np.zeros(len(ACTION_SPACE)))
    return q_table


def choose_action(state, q_table, epsilon):
    """Chooses an action based on an epsilon-greedy policy."""
    if np.random.random() < epsilon:
        return np.random.choice(ACTION_SPACE)
    else:
        action_index = np.argmax(q_table[state])
        return REVERSE_ACTION_MAP[action_index]


def update_q_table(q_table, state, action, reward, next_state, alpha, gamma):
    """Updates the Q-table using the Q-learning update rule."""
    action_index = ACTION_MAP[action]
    best_next_action = np.argmax(q_table[next_state])
    td_target = reward + gamma * q_table[next_state][best_next_action]
    td_error = td_target - q_table[state][action_index]
    q_table[state][action_index] += alpha * td_error
    return q_table

## Training Loop

In [9]:
def train_agent(df, episodes, alpha, gamma, epsilon_start, epsilon_end):
    """Trains the Q-learning agent."""
    q_table = create_q_table()
    epsilon = epsilon_start

    for episode in range(episodes):
        epsilon = epsilon_start - (episode / episodes) * (epsilon_start - epsilon_end)

        # Iterate through each row in the dataframe (simulating a game)
        for i in range(len(df)):
            row = df.iloc[i]
            state = row['state']

            possible_actions = [action for action in ['H', 'S'] if action in row['actions_taken'][2:-2].replace("'", "").replace(" ", "").split(",")]
            if not possible_actions:
              continue

            action = choose_action(state, q_table, epsilon)

            if action not in possible_actions:
                action = np.random.choice(possible_actions)

            reward = row['reward']

            # Determine the next stat
            if i + 1 < len(df):
                next_row = df.iloc[i + 1]
                next_state = next_row['state']
            else:
                next_state = state

            q_table = update_q_table(q_table, state, action, reward, next_state, alpha, gamma)

        if episode % 10 == 0:
            print(f"Episode: {episode}, Epsilon: {epsilon}")

    return q_table

In [10]:
episodes = 100
alpha = 0.01    # Learning rate
gamma = 0.9     # Discount factor
epsilon_start = 0.3
epsilon_end = 0.01

In [11]:
## Actual Training
q_table = train_agent(df, episodes, alpha, gamma, epsilon_start, epsilon_end)

Episode: 0, Epsilon: 0.3
Episode: 10, Epsilon: 0.271
Episode: 20, Epsilon: 0.242
Episode: 30, Epsilon: 0.213
Episode: 40, Epsilon: 0.184
Episode: 50, Epsilon: 0.155
Episode: 60, Epsilon: 0.126
Episode: 70, Epsilon: 0.097
Episode: 80, Epsilon: 0.068
Episode: 90, Epsilon: 0.03899999999999998


In [12]:
def evaluate_policy(df, q_table, evaluation_episodes=1000):
    """Evaluates the learned policy on a separate set of data (or a subset of the training data)."""
    total_reward = 0
    num_hands = 0

    for i in range(len(df)): # Limit the number of evaluation hands
        row = df.iloc[i]
        state = row['state']

        # Filter actions to only allow 'H' and 'S'
        possible_actions = [action for action in ['H', 'S'] if action in row['actions_taken'][2:-2].replace("'", "").replace(" ", "").split(",")]

        if not possible_actions:
          continue

        action_index = np.argmax(q_table[state])
        action = REVERSE_ACTION_MAP[action_index]

        if action not in possible_actions:
                action = np.random.choice(possible_actions)

        reward = row['reward']
        total_reward += reward
        num_hands += 1

        if num_hands >= evaluation_episodes:
            break


    average_reward = total_reward / num_hands if num_hands > 0 else 0
    print(f"Evaluation over {num_hands} hands: Average Reward = {average_reward}")
    return average_reward


evaluation_episodes = 1000
average_reward = evaluate_policy(df, q_table, evaluation_episodes)
average_reward

Evaluation over 1000 hands: Average Reward = 0.0325


0.0325

In [14]:
import pickle

ACTION_SPACE = ['H', 'S']


def default_q_value():
    """Returns a default Q-value (NumPy array of zeros).  Defined at the module level."""
    return np.zeros(len(ACTION_SPACE))


def create_q_table():
    """Initializes the Q-table."""
    q_table = defaultdict(default_q_value)  # Use the named function
    return q_table


def save_q_table(q_table, filename="blackjack_q_table.pkl"):
    """Saves the Q-table to a file using pickle."""
    with open(filename, 'wb') as f:
        pickle.dump(q_table, f)
    print(f"Q-table saved to {filename}")


q_table = create_q_table()
q_table[(15, 10, False, 0)][0] = 0.1
q_table[(15, 10, False, 0)][1] = 0.2


save_q_table(q_table)

Q-table saved to blackjack_q_table.pkl


In [15]:
def load_q_table(filename="blackjack_q_table.pkl"):
    """Loads a Q-table from a file."""
    try:
        with open(filename, 'rb') as f:
            q_table = pickle.load(f)
        print(f"Q-table loaded from {filename}")
        return q_table
    except FileNotFoundError:
        print(f"Error: File {filename} not found.")
        return None
    
loaded_q_table = load_q_table()
if loaded_q_table:
    average_reward_loaded = evaluate_policy(df, loaded_q_table, evaluation_episodes)
average_reward_loaded

Q-table loaded from blackjack_q_table.pkl
Evaluation over 1000 hands: Average Reward = 0.0325


0.0325